# Demo 3 - Content safety: one policy, two directions (Azure API Management)

## Scenario & talk track

**Content safety is one policy in two directions.** The same `llm-content-safety`
policy element is used **inbound** to check the prompt before it reaches Azure
OpenAI, and **outbound** to check the completion before it reaches the caller:

- **Inbound - prompt checks:** categories, blocklists, prompt shield
  (`shield-prompt="true"` detects prompt-injection / jailbreak attempts).
- **Outbound - completion checks:** windowing, thresholds, stream stop.
  Configured inbound; optionally enforced on completions too. Stress this
  half out loud -- **models can produce unsafe content even from perfectly
  benign prompts**, so checking only the prompt is not enough.

Severity thresholds run **0-7** across four harm categories -- **Hate,
SelfHarm, Sexual, Violence** -- on the `EightSeverityLevels` scale. **Choosing
a threshold is a business decision, not an engineering default.** Involve your
Responsible AI reviewers before changing the defaults below (all four default
to `4`).

We will apply an APIM policy that:

1. Routes to a dedicated Azure OpenAI backend (`demo3-openai-backend`), same
   as Demos 1 and 2.
2. Calls a dedicated Azure AI Content Safety backend
   (`demo3-content-safety-backend`) via `llm-content-safety`, once in
   `<inbound>` (prompt + `enforce-on-completions`) and once in `<outbound>`
   (windowed completion re-check, including the streaming stop behavior).
3. Returns `403` with a clear JSON body and `x-content-safety-decision` /
   `x-content-safety-reason` response headers whenever content is blocked, so
   we have evidence for the test matrix below.


## Demo safety rule

> **Test the gate with a matrix, not with improvisation.** Use versioned,
> pre-approved fixtures from your own evaluation set. **Never improvise
> "harmful" examples live** -- it is a compliance risk and it makes results
> unrepeatable. The fixtures used below (`shared/fixtures.py`) are
> deliberately mild, non-graphic, clearly-labelled placeholders that exercise
> the *mechanism* only. Before running this in your own organization, replace
> them with your own organization's pre-approved evaluation-set fixtures,
> reviewed by your Responsible AI team.

## Demo isolation

Demo 3 reuses the **same Azure API Management instance** and the same Azure
OpenAI / Microsoft Foundry backend values from Demos 1 and 2. It does **not**
create a new APIM instance. It **does** require a new **Azure AI Content
Safety** resource (see Preflight below).

Isolation comes from:

- Dedicated product: `demo3-content-safety`
- Dedicated subscription: `demo3-content-safety-sub`
- Dedicated API ID: `demo3-content-safety-api`
- Dedicated backends: `demo3-openai-backend` and `demo3-content-safety-backend`
- The current run time window and the persisted `DEMO_RUN` suffix sent as
  `x-demo-run` for request traceability.


In [ ]:
import sys
sys.path.append("..")

import re
import time

import requests

from shared import auth, config, apim, display
from shared.fixtures import DEMO3_FIXTURES, FIXTURE_SET_VERSION

display.banner(
    "Demo safety rule: use versioned, pre-approved fixtures from your own evaluation set. "
    "Never improvise harmful examples live -- it is a compliance risk and makes results "
    "unrepeatable. See shared/fixtures.py.",
    kind="warning",
)

cfg = config.load_config(interactive=True)
config.validate_config(cfg)
cfg = config.ensure_content_safety_config(cfg, interactive=True)

DEMO_API_ID = "demo3-content-safety-api"
DEMO_OPENAI_BACKEND_ID = "demo3-openai-backend"
DEMO_CONTENT_SAFETY_BACKEND_ID = "demo3-content-safety-backend"
DEMO_PRODUCT_ID = "demo3-content-safety"
DEMO_SUBSCRIPTION_ID = "demo3-content-safety-sub"
DEMO_PATH = "demo3-content-safety"
DEMO_NAMED_VALUE_AOAI_KEY = "demo3-aoai-key"
DEMO_NAMED_VALUE_CS_KEY = "demo3-content-safety-key"
DEMO_NAMED_VALUE_BLOCKLIST_ID = "demo3-content-safety-blocklist-id"
DEMO_NAMED_VALUE_THRESHOLD_HATE = "demo3-content-safety-threshold-hate"
DEMO_NAMED_VALUE_THRESHOLD_SELFHARM = "demo3-content-safety-threshold-selfharm"
DEMO_NAMED_VALUE_THRESHOLD_SEXUAL = "demo3-content-safety-threshold-sexual"
DEMO_NAMED_VALUE_THRESHOLD_VIOLENCE = "demo3-content-safety-threshold-violence"

DEMO_RUN = cfg.demo_run
API_STYLE = cfg.aoai_api_style

print(f"Demo 3 APIM product/subscription: {DEMO_PRODUCT_ID} / {DEMO_SUBSCRIPTION_ID}")
print(f"Azure OpenAI endpoint: {cfg.aoai_endpoint} (api style: {API_STYLE})")
print(f"Content Safety endpoint: {cfg.content_safety_endpoint}")
print(f"Content Safety key configured: {bool(cfg.content_safety_key)}")
print(
    "Category thresholds (0=most restrictive, 7=least restrictive): "
    f"Hate={cfg.content_safety_threshold_hate}, "
    f"SelfHarm={cfg.content_safety_threshold_selfharm}, "
    f"Sexual={cfg.content_safety_threshold_sexual}, "
    f"Violence={cfg.content_safety_threshold_violence}"
)
print(f"Current DEMO_RUN suffix sent as x-demo-run: {DEMO_RUN}")
print(f"Fixture set version: {FIXTURE_SET_VERSION}")


## Preflight checks (run these out loud)

Before configuring or sending traffic, call out each prerequisite:

1. **APIM instance reachable** -- same instance used in Demos 1 and 2.
2. **Azure OpenAI values present** -- `AOAI_ENDPOINT` / `AOAI_DEPLOYMENT` from Demo 1/2.
3. **Azure AI Content Safety resource present** -- `CONTENT_SAFETY_ENDPOINT` is set,
   in the correct resource-root format, with an optional key.
4. **APIM managed identity has the `Cognitive Services User` role on the Content
   Safety resource** -- required unless `CONTENT_SAFETY_KEY` is supplied instead.


In [ ]:
display.header("Preflight checks")

preflight_rows = []

try:
    service = apim.get_service(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
    preflight_rows.append({
        "check": "APIM instance reachable",
        "status": "PASS",
        "detail": f"'{cfg.apim_name}' reachable, gatewayUrl={service['properties']['gatewayUrl']}",
        "remediation": "",
    })
except Exception as exc:
    preflight_rows.append({
        "check": "APIM instance reachable",
        "status": "FAIL",
        "detail": str(exc),
        "remediation": "Verify APIM_RESOURCE_GROUP / APIM_NAME in .env (same as Demos 1 and 2).",
    })

aoai_ok = bool(cfg.aoai_endpoint and cfg.aoai_deployment)
preflight_rows.append({
    "check": "Azure OpenAI values present",
    "status": "PASS" if aoai_ok else "FAIL",
    "detail": f"endpoint={cfg.aoai_endpoint or '<missing>'}, deployment={cfg.aoai_deployment or '<missing>'}",
    "remediation": "Complete Demo 1 first, or set AOAI_ENDPOINT / AOAI_DEPLOYMENT in .env.",
})

try:
    config.validate_content_safety_config(cfg)
    cs_ok = True
    cs_detail = f"endpoint={cfg.content_safety_endpoint}"
except ValueError as exc:
    cs_ok = False
    cs_detail = str(exc)
preflight_rows.append({
    "check": "Azure AI Content Safety resource present",
    "status": "PASS" if cs_ok else "FAIL",
    "detail": cs_detail,
    "remediation": "Create an Azure AI Content Safety resource and set CONTENT_SAFETY_ENDPOINT in .env.",
})

if cfg.content_safety_key:
    role_status, role_detail = "PASS", "CONTENT_SAFETY_KEY supplied; managed identity role is not required."
else:
    role_status, role_detail = (
        "MANUAL",
        "No CONTENT_SAFETY_KEY supplied -- grant the APIM system-assigned managed identity "
        "the 'Cognitive Services User' role on the Content Safety resource "
        "(Content Safety resource -> Access control (IAM) -> Add role assignment).",
    )
preflight_rows.append({
    "check": "APIM managed identity has Cognitive Services User role",
    "status": role_status,
    "detail": role_detail,
    "remediation": "Portal: Content Safety resource -> Access control (IAM) -> Add role assignment -> Cognitive Services User -> APIM managed identity.",
})

display.show_table(preflight_rows, columns=["check", "status", "detail", "remediation"])
for row in preflight_rows:
    kind = "success" if row["status"] == "PASS" else ("warning" if row["status"] == "MANUAL" else "error")
    display.banner(f"{row['status']}: {row['check']}", kind=kind)


## Configure (policy apply)

This section creates or updates the Demo 3 resources on the existing APIM
instance using idempotent ARM `PUT` calls:

- Backend `demo3-openai-backend` (Azure OpenAI origin)
- Backend `demo3-content-safety-backend` (Content Safety origin -- backend-level
  managed identity auth by default, or an `Ocp-Apim-Subscription-Key` header
  when `CONTENT_SAFETY_KEY` is supplied)
- API `demo3-content-safety-api` at path `/demo3-content-safety`
- `chat-completions` operation (url template switched on `API_STYLE`, as in Demo 2)
- Product `demo3-content-safety` + subscription `demo3-content-safety-sub`
- Named values for the four category thresholds, optional blocklist id, and
  optional `demo3-aoai-key` / `demo3-content-safety-key`
- API-scope policy from `policies/demo3-content-safety.xml`


In [ ]:
display.header("Creating backends")

apim.ensure_backend(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    backend_id=DEMO_OPENAI_BACKEND_ID,
    backend_url=cfg.aoai_endpoint,
    description="Demo 3 Azure OpenAI backend",
    protocol="http",
)

if cfg.content_safety_key:
    cs_credentials = {"header": {"Ocp-Apim-Subscription-Key": [cfg.content_safety_key]}}
    display.banner(
        "Content Safety backend will authenticate with an Ocp-Apim-Subscription-Key header (key masked).",
        kind="info",
    )
else:
    cs_credentials = {"managedIdentity": {"resource": "https://cognitiveservices.azure.com"}}
    display.banner(
        "Content Safety backend will authenticate with the APIM managed identity "
        "(grant it 'Cognitive Services User' on the Content Safety resource).",
        kind="info",
    )

apim.ensure_backend(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    backend_id=DEMO_CONTENT_SAFETY_BACKEND_ID,
    backend_url=cfg.content_safety_endpoint,
    description="Demo 3 Azure AI Content Safety backend",
    protocol="http",
    credentials=cs_credentials,
)
display.banner(
    f"Backends '{DEMO_OPENAI_BACKEND_ID}' and '{DEMO_CONTENT_SAFETY_BACKEND_ID}' ensured.",
    kind="success",
)


In [ ]:
display.header("Creating API and operation")

api = apim.ensure_api(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
    display_name="Demo 3 - Content Safety (Azure OpenAI)",
    path=DEMO_PATH,
    service_url=cfg.aoai_endpoint.rstrip("/"),
    subscription_required=True,
)

if API_STYLE == "v1":
    OPERATION_URL_TEMPLATE = "/openai/v1/chat/completions"
else:
    OPERATION_URL_TEMPLATE = f"/openai/deployments/{cfg.aoai_deployment}/chat/completions"

apim.ensure_operation(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
    operation_id="chat-completions",
    display_name="Chat Completions",
    method="POST",
    url_template=OPERATION_URL_TEMPLATE,
)
display.banner(f"API '{DEMO_API_ID}' and operation '{OPERATION_URL_TEMPLATE}' ensured.", kind="success")


In [ ]:
display.header("Creating product, subscription, and named values")

apim.ensure_product(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    product_id=DEMO_PRODUCT_ID,
    display_name="Demo3-Content-Safety",
    description="Isolated product for the Demo 3 content-safety workshop scenario.",
    subscription_required=True,
    state="published",
)
apim.ensure_product_api_link(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    product_id=DEMO_PRODUCT_ID,
    api_id=DEMO_API_ID,
)
apim.ensure_subscription(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    apim_subscription_id=DEMO_SUBSCRIPTION_ID,
    display_name="Demo3-Content-Safety-Subscription",
    scope=f"/products/{DEMO_PRODUCT_ID}",
)

if cfg.aoai_key:
    apim.ensure_named_value(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name,
        named_value_id=DEMO_NAMED_VALUE_AOAI_KEY,
        display_name=DEMO_NAMED_VALUE_AOAI_KEY,
        value=cfg.aoai_key,
        secret=True,
    )
    display.banner("AOAI key named value ensured (key masked).", kind="success")
else:
    display.banner(
        "No AOAI key supplied -- assuming managed identity is configured for this APIM instance to call Azure OpenAI.",
        kind="warning",
    )

if cfg.content_safety_key:
    apim.ensure_named_value(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name,
        named_value_id=DEMO_NAMED_VALUE_CS_KEY,
        display_name=DEMO_NAMED_VALUE_CS_KEY,
        value=cfg.content_safety_key,
        secret=True,
    )
    display.banner("Content Safety key named value ensured (key masked).", kind="success")

if cfg.content_safety_blocklist_id:
    apim.ensure_named_value(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name,
        named_value_id=DEMO_NAMED_VALUE_BLOCKLIST_ID,
        display_name=DEMO_NAMED_VALUE_BLOCKLIST_ID,
        value=cfg.content_safety_blocklist_id,
        secret=False,
    )
    display.banner("Content Safety blocklist id named value ensured.", kind="success")
else:
    display.banner(
        "No CONTENT_SAFETY_BLOCKLIST_ID supplied -- the policy's <blocklists> element stays commented out.",
        kind="info",
    )

for named_value_id, threshold in (
    (DEMO_NAMED_VALUE_THRESHOLD_HATE, cfg.content_safety_threshold_hate),
    (DEMO_NAMED_VALUE_THRESHOLD_SELFHARM, cfg.content_safety_threshold_selfharm),
    (DEMO_NAMED_VALUE_THRESHOLD_SEXUAL, cfg.content_safety_threshold_sexual),
    (DEMO_NAMED_VALUE_THRESHOLD_VIOLENCE, cfg.content_safety_threshold_violence),
):
    apim.ensure_named_value(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name,
        named_value_id=named_value_id,
        display_name=named_value_id,
        value=str(threshold),
        secret=False,
    )

display.banner(
    f"Product '{DEMO_PRODUCT_ID}', subscription '{DEMO_SUBSCRIPTION_ID}', and threshold named values ensured.",
    kind="success",
)


## Apply the policy at API scope

Below is the full policy XML from `policies/demo3-content-safety.xml`. It
uses `llm-content-safety` **twice**: once in `<inbound>` (prompt checks +
`enforce-on-completions`) and once in `<outbound>` (windowed completion
checks, including the streaming-stop behavior). The AOAI backend call
authenticates with the APIM managed identity by default; if `AOAI_KEY` is
set, the notebook swaps that block for an `api-key` header sourced from the
`demo3-aoai-key` named value, exactly as in Demo 1/2. (The Content Safety
backend's own authentication is configured on the backend entity itself, set
above -- it does not need a similar swap in this policy XML.)


In [ ]:
with open("../policies/demo3-content-safety.xml", encoding="utf-8-sig") as f:
    policy_xml = f.read()

_MI_AUTH_BLOCK = re.compile(
    r"\s*<authentication-managed-identity\b.*?</set-header>", re.DOTALL
)
_API_KEY_BLOCK = (
    '\n    <set-header name="api-key" exists-action="override">'
    "\n      <value>{{demo3-aoai-key}}</value>"
    "\n    </set-header>"
)

if cfg.aoai_key:
    policy_xml, replaced = _MI_AUTH_BLOCK.subn(_API_KEY_BLOCK, policy_xml, count=1)
    if not replaced:
        raise ValueError("Could not find the managed-identity block in the policy XML.")
    display.banner(
        "AOAI key supplied -- the policy will authenticate to the Azure OpenAI backend with the 'api-key' header (value from demo3-aoai-key).",
        kind="info",
    )
else:
    display.banner(
        "No AOAI key -- the policy authenticates to the Azure OpenAI backend with the APIM managed identity.",
        kind="info",
    )

print(policy_xml)


In [ ]:
apim.set_api_policy(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
    policy_xml=policy_xml,
)
display.banner("Policy applied at API scope.", kind="success")


## Fixtures: the test matrix, not toxic improvisation

`shared/fixtures.py` defines a single, versioned set of fixtures for this
demo (`FIXTURE_SET_VERSION`). Per the demo safety rule above, the
`prompt_injection`, `harm_threshold`, and `streaming_completion` entries are
**mild, non-graphic, clearly-labelled placeholders** -- they exist to
demonstrate the *mechanism*, not to ship toxic content. **Substitute your own
organization's pre-approved evaluation-set fixtures** before using this in a
real workshop or rollout.


In [ ]:
display.header(f"Demo 3 fixtures (version {FIXTURE_SET_VERSION})")
_ = display.show_table(
    list(DEMO3_FIXTURES.values()),
    columns=["case", "input_label", "expected_status", "expected_evidence"],
)


## Data-plane call helper

Every call goes through the APIM gateway with the dedicated Demo 3
subscription key, an `x-demo-run` header for traceability, and captures
status, the content-safety evidence headers, and latency.


In [ ]:
GATEWAY_URL = apim.get_gateway_url(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
SUBSCRIPTION_KEY = apim.get_subscription_key(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_SUBSCRIPTION_ID
)

print(f"Gateway URL: {GATEWAY_URL}")
print(f"Subscription key: {auth.mask_secret(SUBSCRIPTION_KEY)}")


def _chat_url_and_body(prompt: str, max_tokens: int = 200, stream: bool = False):
    if API_STYLE == "v1":
        url = f"{GATEWAY_URL}/{DEMO_PATH}/openai/v1/chat/completions"
        body = {
            "model": cfg.aoai_deployment,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
        }
    else:
        url = (
            f"{GATEWAY_URL}/{DEMO_PATH}/openai/deployments/"
            f"{cfg.aoai_deployment}/chat/completions"
        )
        body = {
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
        }
    if stream:
        body["stream"] = True
    return url, body


def call_chat_completion(prompt: str, max_tokens: int = 200):
    """Call Demo 3 once (non-streaming) and capture content-safety evidence."""
    headers = {
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY,
        "Content-Type": "application/json",
        "x-demo-run": DEMO_RUN,
    }
    url, body = _chat_url_and_body(prompt, max_tokens=max_tokens)
    params = {} if API_STYLE == "v1" else {"api-version": cfg.aoai_api_version}

    start = time.time()
    response = requests.post(url, headers=headers, params=params, json=body, timeout=90)
    latency_ms = round((time.time() - start) * 1000, 1)

    reply = None
    if response.status_code == 200:
        try:
            reply = response.json()["choices"][0]["message"]["content"]
        except Exception:
            reply = None

    return {
        "status": response.status_code,
        "latency_ms": latency_ms,
        "decision": response.headers.get("x-content-safety-decision"),
        "reason": response.headers.get("x-content-safety-reason"),
        "reply": reply,
        "body": response.text[:2000],
        "url": response.url,
    }


def explain_failure(result):
    status = result["status"]
    display.banner(f"Call did not return the expected status (got {status}): {result['body']}", kind="error")
    if status == 404:
        print(f"Requested URL   : {result.get('url')}")
        print("Likely causes: AOAI_ENDPOINT includes a path, AOAI_DEPLOYMENT is wrong, or AOAI_API_STYLE is wrong.")
    elif status in (401,):
        print("Likely cause: APIM managed identity lacks 'Cognitive Services OpenAI User' on Azure OpenAI, or AOAI_KEY is invalid.")
    elif status == 500:
        print(
            "Likely cause: a policy expression/runtime error, or the Content Safety backend's managed identity "
            "is missing the 'Cognitive Services User' role (or CONTENT_SAFETY_KEY is invalid)."
        )


## Run the test matrix

Call the gateway once per non-streaming case in the test matrix: **safe
business prompt** (expect `200`), **prompt attack** (expect `403` via prompt
shield), and **harm threshold** (expect `403` via the category policy). The
streaming case is handled separately below because "stream stops" is not a
status code.

> The shipped harm-category fixture is intentionally mild and may score below
the configured threshold. A **NOT TRIPPED** result is a prompt to use your
organization's pre-approved evaluation-set fixture or temporarily lower the
relevant category threshold for the demonstration; it is not a gateway error.


In [ ]:
matrix_results = {}

for key in ("safe_business_prompt", "prompt_injection", "harm_threshold"):
    fixture = DEMO3_FIXTURES[key]
    result = call_chat_completion(fixture["prompt"])
    matrix_results[key] = result
    if result["status"] not in (200, 403):
        explain_failure(result)

matrix_rows = []
for key, fixture in DEMO3_FIXTURES.items():
    if key == "streaming_completion":
        continue
    result = matrix_results[key]
    passed = result["status"] == fixture["expected_status"]
    not_tripped = key == "harm_threshold" and result["status"] == 200
    outcome = "PASS" if passed else "NOT TRIPPED" if not_tripped else "FAIL"
    matrix_rows.append({
        "Case": fixture["case"],
        "Input": fixture["input_label"],
        "Expected": fixture["expected_status"],
        "Actual": result["status"],
        "Evidence": (
            f"decision={result['decision']}, reason={result['reason']}"
            if result["status"] == 403
            else "Fixture scored below the configured category threshold"
            if not_tripped
            else fixture["expected_evidence"]
        ),
        "Pass/Fail": outcome,
    })

_ = display.show_table(matrix_rows, columns=["Case", "Input", "Expected", "Actual", "Evidence", "Pass/Fail"])
for row in matrix_rows:
    outcome = row["Pass/Fail"]
    if outcome == "NOT TRIPPED":
        display.banner(
            "NOT TRIPPED: the placeholder fixture is intentionally mild and may score below the configured threshold. "
            "Substitute your organization's pre-approved evaluation-set fixture. Alternatively, the relevant category "
            "threshold named value may be too high; lower CONTENT_SAFETY_THRESHOLD_VIOLENCE toward the low end of the "
            "0-7 scale to demonstrate the mechanism. The production threshold is a Responsible AI business decision, "
            "not an engineering default.",
            kind="warning",
        )
    else:
        display.banner(
            f"{outcome}: {row['Case']}",
            kind="success" if outcome == "PASS" else "error",
        )


## Streaming case: outbound stream stop

Per Microsoft's documented behavior, when `llm-content-safety` is applied in
`<outbound>` to a streaming response, a detected violation causes APIM to
**stop forwarding further events to the client -- it does not return a 403**
for this case. We demonstrate this by streaming the controlled fixture and
showing that the stream ends **without** the terminal `[DONE]` event, i.e.
"no later events forwarded".

> Because the streaming fixture is intentionally mild, a completed stream may
be **NOT TRIPPED** at the configured threshold. Use a pre-approved evaluation-set
fixture or temporarily lower the relevant category threshold to demonstrate the
stream-stop mechanism reliably.


In [ ]:
def call_streaming_chat_completion(prompt: str, max_tokens: int = 300):
    headers = {
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY,
        "Content-Type": "application/json",
        "x-demo-run": DEMO_RUN,
    }
    url, body = _chat_url_and_body(prompt, max_tokens=max_tokens, stream=True)
    params = {} if API_STYLE == "v1" else {"api-version": cfg.aoai_api_version}

    start = time.time()
    response = requests.post(url, headers=headers, params=params, json=body, timeout=90, stream=True)

    events = []
    saw_done = False
    for raw_line in response.iter_lines(decode_unicode=True):
        if not raw_line:
            continue
        events.append(raw_line)
        if raw_line.strip() == "data: [DONE]":
            saw_done = True
            break

    latency_ms = round((time.time() - start) * 1000, 1)
    return {
        "status": response.status_code,
        "latency_ms": latency_ms,
        "events": events,
        "event_count": len(events),
        "saw_done": saw_done,
        "stream_stopped_early": response.status_code == 200 and not saw_done,
    }


streaming_fixture = DEMO3_FIXTURES["streaming_completion"]
streaming_result = call_streaming_chat_completion(streaming_fixture["prompt"])

display.show_table([{
    "status": streaming_result["status"],
    "latency_ms": streaming_result["latency_ms"],
    "event_count": streaming_result["event_count"],
    "saw_[DONE]": streaming_result["saw_done"],
}])

print("Events received before the stream ended:")
for event in streaming_result["events"]:
    print(event)

streaming_pass = streaming_result["stream_stopped_early"]
streaming_not_tripped = streaming_result["status"] == 200 and streaming_result["saw_done"]
if streaming_pass:
    streaming_outcome = "PASS"
    display.banner("PASS: stream stopped early, no [DONE] event forwarded.", kind="success")
elif streaming_not_tripped:
    streaming_outcome = "NOT TRIPPED"
    display.banner(
        "NOT TRIPPED: the placeholder fixture is intentionally mild and may score below the configured threshold. "
        "Substitute your organization's pre-approved evaluation-set fixture. Alternatively, the relevant category "
        "threshold named value may be too high; lower CONTENT_SAFETY_THRESHOLD_VIOLENCE toward the low end of the "
        "0-7 scale to demonstrate the mechanism. The production threshold is a Responsible AI business decision, "
        "not an engineering default.",
        kind="warning",
    )
else:
    streaming_outcome = "FAIL"
    display.banner(
        f"FAIL: streaming request returned unexpected gateway status {streaming_result['status']}.",
        kind="error",
    )

matrix_rows.append({
    "Case": streaming_fixture["case"],
    "Input": streaming_fixture["input_label"],
    "Expected": streaming_fixture["expected_status"],
    "Actual": (
        "STREAM STOPPED" if streaming_pass
        else "STREAM COMPLETED" if streaming_not_tripped
        else f"HTTP {streaming_result['status']}"
    ),
    "Evidence": f"{streaming_result['event_count']} events forwarded, saw_[DONE]={streaming_result['saw_done']}",
    "Pass/Fail": streaming_outcome,
})
_ = display.show_table(matrix_rows, columns=["Case", "Input", "Expected", "Actual", "Evidence", "Pass/Fail"])


## Summary: what you saw -> which policy/knob made it happen

| What you saw | Which policy/knob made it happen |
| --- | --- |
| Safe business prompt succeeds (`200`) | Content scored below every category threshold and no prompt-shield match |
| Prompt attack blocked (`403`) | `shield-prompt="true"` in the inbound `llm-content-safety` |
| Harm-threshold fixture blocked (`403`) | `<categories output-type="EightSeverityLevels">` thresholds, sourced from named values |
| Streaming completion stops early, no later events forwarded | Outbound `llm-content-safety` with `window-size` / `window-overlap-size`, documented stream-stop behavior (no 403 for this case) |
| Clear 403 evidence (`x-content-safety-decision`, `x-content-safety-reason`, JSON body) | `<on-error>` handling `context.LastError.Source == "llm-content-safety"` |

**Two directions, one policy:** the same `llm-content-safety` element is used
inbound (prompt) and outbound (completion) -- content safety is not solely a
prompt-filtering concern. **Thresholds are a business decision, not an
engineering default**; the 0-7 `EightSeverityLevels` scale should be tuned
with your Responsible AI reviewers, not left at whatever an engineer typed
in. **Fixture discipline matters**: every fixture used above is versioned,
pre-approved, and mild/non-graphic -- never improvise harmful examples live.

Demo 3 leaves its APIM resources in place -- Demo 4 reuses the same APIM
instance, so cleanup is covered at the end of the final demo.

Re-running `demo3-content-safety.ipynb` end to end, twice in a row, does not
fail or duplicate any Azure resources.
